In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

query = """
SELECT customer_id, feature_date, f_30, m_30, recency_days
FROM hpce.customer_features_daily
WHERE feature_date BETWEEN '2018-06-01' AND '2018-08-04';
"""
df = pd.read_sql(query, engine)
df["feature_date"] = pd.to_datetime(df["feature_date"])

# Engagement score
f = np.log1p(df["f_30"].fillna(0))
m = np.log1p(df["m_30"].fillna(0))
r = df["recency_days"].fillna(999).clip(0, 999)

# Normalize to 0..1
f = (f - f.min()) / (f.max() - f.min() + 1e-9)
m = (m - m.min()) / (m.max() - m.min() + 1e-9)
r = (r - r.min()) / (r.max() - r.min() + 1e-9)

eng = (0.45 * f + 0.45 * m + 0.10 * (1 - r)).clip(0, 1)
df["engagement"] = eng

rng = np.random.default_rng(42)

# Medium volume Poisson rates
lam_view   = 0.8 + 6.0 * df["engagement"]     # ~1 to ~7
lam_search = 0.2 + 1.8 * df["engagement"]     # ~0 to ~2
lam_cart   = 0.05 + 0.7 * df["engagement"]    # ~0 to ~1

# mild weekend boost
dow = df["feature_date"].dt.dayofweek
season = np.where(dow >= 5, 1.2, 1.0)
lam_view *= season
lam_search *= season
lam_cart *= season

df["n_view"] = rng.poisson(lam_view)
df["n_search"] = rng.poisson(lam_search)
df["n_cart"] = rng.poisson(lam_cart)

def random_times(day, n):
    secs = rng.integers(0, 24*3600, size=n)
    return day + pd.to_timedelta(secs, unit="s")

rows = []
for _, row in df.iterrows():
    day = row["feature_date"]
    cid = row["customer_id"]

    for etype, ncol in [("view_product", "n_view"), ("search", "n_search"), ("add_to_cart", "n_cart")]:
        n = int(row[ncol])
        if n <= 0:
            continue
        for t in random_times(day, n):
            rows.append({
                "customer_id": cid,
                "event_time": t.to_pydatetime(),
                "event_type": etype,
                "session_id": None,
                "source": "synthetic",
                "meta": None
            })

events = pd.DataFrame(rows)
print("Synthetic events rows:", len(events))

events.to_sql(
    "synthetic_events_raw",
    engine,
    schema="hpce",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

Synthetic events rows: 126074


126074